## Import

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
from sindex.metrics.citations import (
    merge_citations_dicts, 
    merge_citations_from_files, 
    merge_citations_from_files_fast,
    combine_citations,
)
from sindex.metrics.mentions import combine_mentions
from sindex.metrics.batch_jobs import batch_dataset_report_metadata
from sindex.metrics.fairscores import extrapolate_emdb_fair_scores
from sindex.utils.files import combine_ndjson_files, merge_ndjson_files_in_folder
import os

## Citations

### Deduplicate citations from different sources

#### DataCite (DOIs)

In [24]:
mdc_citations = r"D:\pipeline-data\citations\mdc\mdc_citations.ndjson"
oa_citations = r"D:\pipeline-data\citations\openalex\oa_citations.ndjson"
dc_citations = r"D:\pipeline-data\citations\datacite\dc_citations.ndjson"
citation_files = [mdc_citations, oa_citations, dc_citations]
output_file_doi = r"D:\pipeline-data\citations\doi_citations.ndjson"

In [25]:
merge_citations_from_files_fast(citation_files, output_file_doi)

Starting merge of 3 valid files...
Finished processing 8,861,682 records. Unique: 7,654,146
Writing to D:\pipeline-data\citations\doi_citations.ndjson...
Done!


#### EMDB

In [17]:
mdc_citations = r"I:\pipeline-data\citations\mdc\mdc_citations_emdb.ndjson"
citation_files = [mdc_citations]
output_file_emdb = r"I:\pipeline-data\citations\emdb_citations.ndjson"

In [18]:
merge_citations_from_files_fast(citation_files, output_file_emdb)

Starting merge of 1 valid files...
Finished processing 15,134 records. Unique: 15,134
Writing to I:\pipeline-data\citations\emdb_citations.ndjson...
Done!


### Combine and add placeholder dates when citation date missing

In [26]:
doi = r"D:\pipeline-data\citations\doi_citations.ndjson"
emdb = r"D:\pipeline-data\citations\emdb_citations.ndjson"
file_list = [doi, emdb]
output_path = r"D:\pipeline-data\citations\citations.ndjson"

In [27]:
combine_ndjson_files(file_list, output_path)

Lines processed: 7,660,000
Finished! Total entries saved: 7,669,280


## Mentions

In [23]:
# Mock mentions file
input_file =  r"I:\pipeline-data\citations\citations.ndjson"
output_file =  r"I:\pipeline-data\mentions\mentions_github_mock.ndjson"

with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        if not line.strip(): continue  # Skip empty lines
        data = json.loads(line)
        
        # Mapping logic: Rename only if key exists
        mapping = {
            'citation_link': 'mention_link',
            'citation_date': 'mention_date',
            'citation_weight': 'mention_weight'
        }
        
        for old_key, new_key in mapping.items():
            if old_key in data:
                data[new_key] = data.pop(old_key)
        
        outfile.write(json.dumps(data) + '\n')

### Combine and add placeholder dates when mention date missing

In [12]:
mock = r"D:\pipeline-data\mentions\mentions_github_mock.ndjson"
file_list = [mock]
output_path = r"D:\pipeline-data\mentions\mentions.ndjson"

In [13]:
combine_mentions(file_list, output_path)

Lines processed: 4,150,000
Finished! Total entries saved: 4,156,510


## FAIR scores

### Merge DOI FAIR score into one file

In [39]:
fair_scores_directory = r"D:\pipeline-data\fair_scores\fair_scores_doi_files"
doi_fair_scores_path = r"D:\pipeline-data\fair_scores\doi_fair_scores.ndjson"

In [40]:
merge_ndjson_files_in_folder(fair_scores_directory, doi_fair_scores_path)

Found 4901 files in 'D:\pipeline-data\fair_scores\fair_scores_doi_files'. Starting merge...
 Complete! Total lines in 'D:\pipeline-data\fair_scores\doi_fair_scores.ndjson': 49,009,521    


### Extrapolate fair scores for EMDB since they are the same

In [29]:
emdb_file_path = r"D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson"
partial_score_file_path = r"D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson"
emdb_fair_scores_path = r"D:\pipeline-data\fair_scores\emdb_fair_scores.ndjson"

In [30]:
extrapolate_emdb_fair_scores(emdb_file_path, partial_score_file_path, emdb_fair_scores_path)

Loading scores from D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson...
Processing D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson...
----------------------------------------
STATISTICS (orjson)
----------------------------------------
Total records in EMDB file:   51645
Total records written:        51645
  - Found existing scores:    18132
  - Extrapolated scores:      33513
  - Skipped (no ID):          0
----------------------------------------
SUCCESS: Input count matches output count.


### Merge all FAIR score into one file

In [41]:
fair_scores_path = r"D:\pipeline-data\fair_scores\fair_scores.ndjson"

In [42]:
combine_ndjson_files([doi_fair_scores_path, emdb_fair_scores_path], fair_scores_path)

Lines processed: 49,000,000
Finished! Total entries saved: 49,061,166


## Topics

In [16]:
# Nothing to process

## Dataset report

### Dataset metadata

In [18]:
slim_folder = r"D:\pipeline-data\records\slim-records"
dataset_report_metadata_file = r"D:\pipeline-data\dataset_index\dataset_report_metadata.ndjson"

In [19]:
batch_dataset_report_metadata(slim_folder, dataset_report_metadata_file)

Scanning files in D:\pipeline-data\records\slim-records...
Found 772 files. Starting processing with 32 cores...


NameError: name 'process_single_file' is not defined

### Citations

In [ ]:
citations_file =  r"I:\pipeline-data\citations\citations.ndjson"
dataset_report_with_citations = r"D:\pipeline-data\dataset_index\dataset_report_w_citations.ndjson"

In [ ]:
dataset_report_add_citations(dataset_report_metadata_file, citations_file, 'final_dataset_metrics.ndjson')

## Normalization factors

## Dataset Index

## S-index